In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('movies_metadata.csv')

In [3]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

this is the main of the things the ejbvbf g\= 

In [5]:
df.shape

(45466, 24)

In [6]:
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [7]:
df = df.drop_duplicates().reset_index(drop=True)

In [8]:
df.shape

(45453, 24)

In [9]:
df = df[['title', 'overview', 'genres','tagline','vote_average', 'popularity']]

In [10]:
df.head(2)

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",NaN,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Roll the dice and unleash the excitement!,6.9,17.015539


In [11]:
df.shape

(45453, 6)

In [12]:
df.isnull().sum()

title               6
overview          954
genres              0
tagline         25045
vote_average        6
popularity          5
dtype: int64

In [13]:
df = df.dropna(subset=['title'])

In [14]:
df['overview'] = df['overview'].fillna('')

In [15]:
df.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [16]:
import ast
df['genres'] = df['genres'].apply(lambda x:"".join([i['name'] for i in ast.literal_eval(x)]))

In [17]:
df.head(2)

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",AnimationComedyFamily,NaN,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,AdventureFantasyFamily,Roll the dice and unleash the excitement!,6.9,17.015539


In [18]:
df['tagline'] = df['tagline'].fillna('')

In [19]:
df['tags']  = df['overview'] + ' ' + df['genres'] + ' ' + df['tagline']

In [20]:
df['tags'].head(2)

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
Name: tags, dtype: object

# Now We are shifting to the NLP 

In [21]:
df['tags'][0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. AnimationComedyFamily "

In [22]:
%pip install nltk


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [24]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\happy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\happy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [25]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [26]:
def perprocess_text(text):
    #Lowercase 
    text = str(text).lower()
    
    #remove the punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    words = text.split()
    # remove stopwords
    words  =[word for word in words if word not in stop_words]
    
    # lamatize the words
    words = [lemmatizer.lemmatize(words) for words in words ]
    
    return ' '.join(words)


In [27]:
df['tags'] = df['tags'].apply(perprocess_text)

In [28]:
df['tags'][1]

'sibling judy peter discover enchanted board game open door magical world unwittingly invite alan adult who trapped inside game year living room alans hope freedom finish game prof risky three find running giant rhinoceros evil monkey terrifying creature adventurefantasyfamily roll dice unleash excitement'

In [29]:
df = df.reset_index(drop=True)

In [30]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45442
Century of Birthing            45443
Betrayal                       45444
Satan Triumphant               45445
Queerama                       45446
Length: 45447, dtype: int64

# Vactorization

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [32]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')

In [33]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [34]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1014462 stored elements and shape (45447, 5000)>

In [35]:
# cosine silimilarity

from sklearn.metrics.pairwise import cosine_similarity

In [36]:
def recommend(title, n=10):
    if title not in indices:
        return "Movie not found in the dataset."
    idx = indices[title]
    sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_idx = sim_score.argsort()[-n-1:-1][::-1]
    return df['title'].iloc[similar_idx].tolist()

In [37]:
recommend('Avatar', 5)

['Apollo 18',
 'The War of the Robots',
 'Project Moon Base',
 'Hercules Against the Moon Men',
 'Unstable Fables: Tortoise vs. Hare']

In [38]:
import pickle
pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl', 'wb'))
pickle.dump(indices, open('indices.pkl', 'wb'))

df.to_pickle('movies_df.pkl')
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))